# **03 - Question Formulation**

## **0. Tổng quan kế hoạch 6 câu hỏi nghiên cứu và phân công phụ trách**

Nhóm gồm **3 thành viên ($n = 3$)**, tương ứng xây dựng **$2 \times n = 6$ câu hỏi nghiên cứu**, trong đó mỗi thành viên sở hữu chính xác **2 câu hỏi**. Bộ câu hỏi bao gồm:
- **Nhóm 1 — Phân tích thống kê chuyên sâu (Câu hỏi 1 – 4, Non-ML):** Mỗi câu hỏi tích hợp từ **4 đến 5 tiêu chí chiều sâu**, và cả nhóm bao phủ **trọn vẹn 6/6 tiêu chí `D1–D6`** của đề bài.
- **Nhóm 2 — Học máy có giám sát (Câu hỏi 5 – 6, Machine Learning):** Gồm 1 bài toán **Phân loại (Classification)** với chia tách theo nhóm người dùng (`GroupShuffleSplit` + `GroupKFold`) và 1 bài toán **Hồi quy (Regression)** với chia tách theo thời gian (`Temporal Split` + `TimeSeriesSplit`), tuân thủ nghiêm ngặt quy tắc chống rò rỉ dữ liệu (*data leakage*) và đóng gói `scikit-learn Pipeline`.

| Mã | Thành viên phụ trách | Câu hỏi nghiên cứu trọng tâm | Đối tượng & Quyết định hỗ trợ | Tiêu chí chiều sâu / ML |
|---|---|---|---|---|
| **Q1** | **Trần Trọng Tú** (`24122024`) | Độ dài cam kết thời gian (`total_runtime_hours`, số tập) tác động thế nào đến tỷ lệ bỏ dở thực tế (`active_drop_rate`) và thời điểm bỏ cuộc sau khi kiểm soát định dạng & thể loại? | Ủy ban sản xuất & nền tảng streaming quyết định cấu trúc mùa chiếu (*1-cour* 12 tập hay *multi-cour* 24+ tập) | **D1, D3, D4, D5, D6** |
| **Q2** | **Phạm Phương** | Mặt bằng điểm số cá nhân và tỷ lệ hoàn thành dịch chuyển ra sao giai đoạn 2006–2018, và hiện tượng anime mới điểm cao đến từ chất lượng hay độ trễ khám phá (`watch_lag_years`)? | Đội ngũ xếp hạng & biên tập nền tảng hiệu chỉnh thuật toán bảng xếp hạng và chọn mùa phát hành chiến lược | **D1, D2, D3, D4, D5** |
| **Q3** | **Cao Bản** | Các phần tiếp theo (`Direct Sequel`) vượt trội về điểm số và giữ chân khán giả đến mức nào so với phần gốc, và bao nhiêu phần trăm chênh lệch là do hiệu ứng *tự chọn lọc khán giả* trên cùng tài khoản? | Nhà đầu tư & Studio phân bổ vốn giữa sản xuất mùa tiếp theo (*Sequel*) và phát triển IP mới (*Standalone / Original*) | **D1, D3, D4, D5, D6** |
| **Q4** | **Cao Bản** | Độ hào phóng chấm điểm (`user_score_leniency`) và tỷ lệ hoàn thành thay đổi thế nào theo bậc thâm niên xem anime và nhân khẩu học (`gender`, nhóm tuổi)? | Bộ phận Cá nhân hóa & Marketing thiết kế luồng gợi ý cho người dùng mới (*cold-start*) so với người xem kỳ cựu (*power watchers*) | **D1, D2, D3, D4, D5, D6** |
| **Q5** | **Trần Trọng Tú** (`24122024`) | **(ML – Classification)** Dự đoán xác suất một người dùng sẽ **bỏ dở (`Dropped`)** thay vì **hoàn thành (`Completed`)** một bộ anime TV dựa trên thông tin trước khi xem | Hệ thống Recommendation hạ hạng gợi ý có rủi ro bỏ dở cao hoặc gắn nhãn cảnh báo nhịp độ cho từng người dùng | **ML** (`GroupKFold`, Leave-one-out, Pipeline, PR-AUC) |
| **Q6** | **Phạm Phương** | **(ML – Regression)** Dự đoán điểm cộng đồng (`score_clean`) của anime sắp ra mắt chỉ từ thông tin tiền sản xuất (*pre-release metadata*) | Bộ phận Mua bản quyền (*Licensing*) định giá hợp đồng bản quyền tối thiểu trước khi mùa phim lên sóng | **ML** (`TimeSeriesSplit` $\le 2015 \to 2016\text{–}2018$, Pipeline, RMSE) |

## **1. Câu hỏi 1 (Non-ML) — Phụ trách: Trần Trọng Tú**

### **1.1. Câu hỏi nghiên cứu**

#### **1.1.1. Nội dung câu hỏi**
Ở giai đoạn Khám phá dữ liệu (Notebook 02), nhóm phát hiện rằng **24.2%** dòng trong `UserAnimeList` mang trạng thái `plan to watch` (chưa từng xem), khiến mọi phép tính tỷ lệ bỏ dở trên tổng danh sách bị sai lệch mẫu số. Khi chỉ xét những người đã thực sự bắt đầu xem, nhóm đặt câu hỏi:

***"Trong nhóm người dùng đã thực sự bắt đầu xem một bộ anime (`my_status_clean` $\in \{1, 2, 3, 4\}$), tổng thời lượng cam kết (`total_runtime_hours` và quy mô số tập) ảnh hưởng như thế nào đến tỷ lệ bỏ dở thực tế (`active_drop_rate`) cũng như tiến độ bỏ cuộc sớm (`early25_drop_share`), và xu hướng này có còn giữ nguyên sau khi kiểm soát định dạng phát hành (`type_clean`) cùng thể loại chính hay không?"***

#### **1.1.2. Làm rõ phương pháp và các tiêu chí chiều sâu (D1, D3, D4, D5, D6)**
- **D1 (Combining data):** Tổng hợp trạng thái xem và số tập đã xem khi bỏ dở từ 80,075,053 dòng `UserAnimeList_clean` theo `anime_id`, sau đó kết hợp (`merge`) với `AnimeList_clean` và kiểm chứng số dòng trước/sau cùng khóa không khớp.
- **D3 (Derived variables):** Xây dựng 4 biến dẫn xuất có cơ sở nghiệp vụ:
  - `active_starters` = tổng số người ở trạng thái `watching (1) + completed (2) + on_hold (3) + dropped (4)` (loại trừ `plan_to_watch (6)`).
  - `active_drop_rate` = `dropped / active_starters`.
  - `total_runtime_hours` = `episodes_clean * duration_minutes_clean / 60`.
  - `drop_progress_ratio` = `my_watched_episodes_clean / episodes_clean` trên các dòng `dropped` (đo thời điểm người xem rời bỏ).
- **D4 (Controlling for a confounder):** Phân tầng riêng cho định dạng truyền hình `TV` (*1-cour* 10–13 tập vs *2-cour* 22–26 tập vs *Long-running* 27+ tập) và kiểm soát trong từng thể loại lớn (`Action`, `Comedy`, `Drama`, `Romance`, `Sci-Fi`, `Slice of Life`).
- **D5 (Quantifying uncertainty):** Tính khoảng tin cậy **Bootstrap 95%** cho tỷ lệ bỏ dở từng phân nhóm, kiểm định **Mann–Whitney U** và tương quan hạng **Spearman**.
- **D6 (Robustness check):** Kiểm tra xem kết luận có đổi chiều khi nâng ngưỡng quy mô khán giả tối thiểu (`active_starters >= 100`, `>= 500`, `>= 1,000`) hoặc khi gộp cả `on_hold` vào định nghĩa ngừng xem (`abandon_rate = (dropped + on_hold) / active_starters`).

### **1.2. Động lực, đối tượng quan tâm và quyết định thực tế**

- **Vì sao đáng nghiên cứu:** Chi phí sản xuất một mùa anime truyền hình tăng theo cấp số nhân với số tập, nhưng nếu kéo dài liên tục 24–50 tập khiến người xem kiệt sức và bỏ cuộc giữa chừng, hiệu quả truyền thông ở nửa sau mùa phim sẽ sụt giảm mạnh.
- **Đối tượng quan tâm (Stakeholders):** Ủy ban sản xuất (*Production Committee*), giám đốc lập lịch phát sóng truyền hình và đội ngũ sản xuất nội dung gốc của nền tảng streaming.
- **Quyết định thực tế được hỗ trợ:** Quyết định nên đặt hàng sản xuất theo mô hình **tách mùa ngắn (*split-cour* 12–13 tập)** thay vì phát hành liền mạch 24+ tập, đồng thời xác định "ngưỡng tập nguy hiểm" (25% thời lượng đầu) để tập trung ngân sách kịch bản và hoạt họa giữ chân người xem.

## **2. Câu hỏi 2 (Non-ML) — Phụ trách: Phạm Phương**

### **2.1. Câu hỏi nghiên cứu**

#### **2.1.1. Nội dung câu hỏi**
Một giả định phổ biến trên các diễn đàn anime là *"khán giả hiện đại chấm điểm hào phóng hơn cho anime mới (recency bias / score inflation)"*. Để kiểm chứng định lượng hiện tượng này trên dữ liệu thực tế giai đoạn 2006–2018, nhóm đặt câu hỏi:

***"Trong giai đoạn 2006–2018, mặt bằng điểm đánh giá cá nhân (`my_score_clean`) và tỷ lệ bỏ dở thay đổi như thế nào theo chu kỳ quý/năm tương tác, và sự chênh lệch điểm số giữa anime các thời kỳ phản ánh lạm phát điểm theo thời gian hay phản ánh độ trễ khám phá (`watch_lag_years` — xem ngay khi vừa chiếu so với xem lại tác phẩm cũ sau nhiều năm)?"***

#### **2.1.2. Làm rõ phương pháp và các tiêu chí chiều sâu (D1, D2, D3, D4, D5)**
- **D1 (Combining data):** Kết hợp hơn 13.1 triệu dòng có ngày/năm hoàn thành hợp lệ từ `UserAnimeList_clean` với năm khởi chiếu `aired_start_year_clean` và định dạng `type_clean` từ `AnimeList_clean`.
- **D2 (Reshaping & time):** Chuỗi thời gian theo Quý (`2006Q1`–`2018Q2`) kèm đường **trung bình trượt 4 quý (*4-quarter rolling average*)**, kết hợp `pivot` bảng ma trận thế hệ (*Cohort Matrix*: Giai đoạn phát sóng × Năm người dùng hoàn thành).
- **D3 (Derived variables):** Xây dựng biến độ trễ thưởng thức `watch_lag_years = my_finish_year_clean - aired_start_year_clean` (chia thành 4 nhóm: Xem ngay trong năm chiếu `0 năm`, `1–2 năm`, `3–5 năm`, và khám phá kinh điển `>5 năm`).
- **D4 (Controlling for a confounder):** Kiểm soát đồng thời định dạng `TV` và nhóm độ trễ `watch_lag_years` để tách hiệu ứng thời điểm chấm điểm khỏi hiệu ứng tuổi đời tác phẩm.
- **D5 (Quantifying uncertainty):** Kiểm định xu hướng đơn điệu theo thời gian bằng tương quan hạng **Spearman** và khoảng tin cậy **Bootstrap 95%** theo từng giai đoạn.

### **2.2. Động lực, đối tượng quan tâm và quyết định thực tế**

- **Vì sao đáng nghiên cứu:** Nếu điểm số thay đổi có hệ thống theo thời điểm xem và độ trễ khám phá, việc xếp hạng trực tiếp bằng trung bình cộng thô sẽ làm thiên lệch bảng xếp hạng toàn thời gian (*All-Time Top Anime*).
- **Đối tượng quan tâm (Stakeholders):** Nhóm kỹ sư thuật toán xếp hạng của MyAnimeList/AniList và bộ phận quản lý danh mục bản quyền (*Catalog Licensing*).
- **Quyết định thực tế được hỗ trợ:** Thiết kế cơ chế chuẩn hóa điểm theo niên đại/độ trễ xem trên bảng xếp hạng và định hướng chiến dịch làm mới danh mục anime kinh điển (*throwback catalog campaigns*).

## **3. Câu hỏi 3 (Non-ML) — Phụ trách: Cao Bản**

### **3.1. Câu hỏi nghiên cứu**

#### **3.1.1. Nội dung câu hỏi**
Khi quan sát bảng xếp hạng MyAnimeList, các mùa phim tiếp theo (*Season 2, Season 3...*) xuất hiện dày đặc ở nhóm điểm cao. Tuy nhiên, chưa rõ điều này đến từ việc phần sau thực sự hay hơn hay do những khán giả không thích đã bỏ cuộc từ phần đầu:

***"Khi giải mã mạng lưới quan hệ từ cột `related` để phân loại anime thành Độc lập (`Standalone`), Phần gốc mở đầu (`Franchise Root`) và Phần tiếp theo trực tiếp (`Direct Sequel`), mức chênh lệch điểm số và quy mô người xem giữa Phần tiếp theo và Phần gốc lớn đến mức nào, và bao nhiêu phần của mức tăng điểm này thực chất do hiệu ứng tự chọn lọc khán giả (*Self-Selection / Survivorship Bias*) trên cùng một tài khoản người dùng?"***

#### **3.1.2. Làm rõ phương pháp và các tiêu chí chiều sâu (D1, D3, D4, D5, D6)**
- **D1 (Combining data):** Phân tách chuỗi từ điển Python trong cột `related` của `AnimeList_clean` thành đồ thị cạnh có hướng `(prequel_id -> sequel_id)`, sau đó kết hợp hai chiều với thống kê toàn bộ 80M dòng và nối trực tiếp điểm chấm cặp `(prequel_score, sequel_score)` trên cùng `username`.
- **D3 (Derived variables):** Tạo phân vai nhượng quyền `franchise_role`, tỷ lệ giữ chân quy mô khán giả `audience_retention_ratio = seq_users / pre_users`, khoảng cách tự chọn lọc `selection_gap` (điểm Phần 1 của người xem tiếp Phần 2 trừ điểm Phần 1 của người dừng lại) và độ lệch nội tại trên cùng người dùng `within_user_delta = sequel_score - prequel_score`.
- **D4 (Controlling for a confounder):** Kiểm soát nguyên tác (`Manga`, `Light novel`, `Original`) và đặc biệt kiểm soát chính cá nhân người chấm thông qua phép đối sánh cặp trên cùng tài khoản (`within-user paired comparison`).
- **D5 (Quantifying uncertainty):** Khoảng tin cậy **Bootstrap 95%**, kiểm định cặp **Wilcoxon signed-rank / paired t-test** và cỡ hiệu ứng **Cohen's $d$**.
- **D6 (Robustness check):** Đối chiếu kết quả giữa tập cặp `TV -> TV` tiêu chuẩn (`scored_by >= 500`) với ngưỡng khắt khe (`scored_by >= 2,500`) và khi mở rộng sang mọi định dạng (`TV/Movie/OVA`).

### **3.2. Động lực, đối tượng quan tâm và quyết định thực tế**

- **Vì sao đáng nghiên cứu:** Đánh đồng "điểm trung bình cao của Sequel" với "thành công thương mại rộng rãi" là cái bẫy đầu tư kinh điển nếu quy mô người xem sụt giảm mạnh qua mỗi mùa.
- **Đối tượng quan tâm (Stakeholders):** Nhà đầu tư ủy ban sản xuất, giám đốc phát triển IP tại các studio hoạt hình.
- **Quyết định thực tế được hỗ trợ:** Cân đối ngân sách giữa việc bật đèn xanh cho *Season 2/3* (tối ưu mức độ hài lòng của tệp fan cốt lõi nhưng quy mô thu hẹp) so với đầu tư chuyển thể IP mới để mở rộng tệp khán giả đầu phễu.

## **4. Câu hỏi 4 (Non-ML) — Phụ trách: Cao Bản**

### **4.1. Câu hỏi nghiên cứu**

#### **4.1.1. Nội dung câu hỏi**
Người dùng trên nền tảng không đồng nhất: có người mới xem vài chục bộ phổ biến, có người đã trải nghiệm hàng trăm tác phẩm ở nhiều độ tuổi và giới tính khác nhau:

***"Độ hào phóng/khắt khe khi chấm điểm (`user_score_leniency` — chênh lệch giữa điểm cá nhân và điểm trung bình cộng đồng của chính bộ anime đó) cùng tỷ lệ hoàn thành thay đổi như thế nào theo bậc thâm niên xem anime (`exp_tier`) và đặc điểm nhân khẩu học (`gender_clean`, nhóm tuổi), sau khi đã kiểm soát chất lượng nội tại của các bộ anime mà họ chọn xem?"***

#### **4.1.2. Làm rõ phương pháp và các tiêu chí chiều sâu (D1, D2, D3, D4, D5, D6)**
- **D1 (Combining data):** Kết hợp bảng hồ sơ `UserList_clean` với hành vi thực tế tính từ `UserAnimeList_clean` và điểm chuẩn từng phim từ `AnimeList_clean` qua `username` và `anime_id`.
- **D2 (Reshaping):** Sử dụng `pivot_table` tạo ma trận đa chiều *Bậc thâm niên × Nhóm tuổi* và *Giới tính × Bậc thâm niên*.
- **D3 (Derived variables):** Xây dựng chỉ số chuẩn hóa `user_score_leniency = my_score_clean - anime_score_clean` (giúp tách biệt người thực sự chấm khắt khe khỏi người chỉ vô tình xem phải phim dở!), bậc kinh nghiệm `exp_tier` (`Casual`, `Regular`, `Avid`, `Power`) và `age_2018`.
- **D4 (Controlling for a confounder):** Kiểm soát chất lượng bộ phim được chọn qua `leniency` và kiểm soát chéo thâm niên xem khi so sánh giữa các nhóm giới tính/độ tuổi.
- **D5 & D6 (Uncertainty & Robustness):** Khoảng tin cậy **Bootstrap 95%**, kiểm định **Kruskal–Wallis** & tương quan **Spearman**; kiểm tra độ bền vững (**D6**) khi loại bỏ tài khoản cực đoan (`extreme_user == False`) và nâng ngưỡng số bộ đã chấm từ $\ge 10$ lên $\ge 30$.

### **4.2. Động lực, đối tượng quan tâm và quyết định thực tế**

- **Vì sao đáng nghiên cứu:** Nếu người xem kỳ cựu chấm điểm khắt khe hơn đáng kể so với người xem mới, các tựa anime kén khán giả (*niche/arthouse*) thu hút nhiều *power watchers* sẽ bị thiệt thòi điểm trung bình so với anime đại chúng dành cho người mới.
- **Đối tượng quan tâm (Stakeholders):** Đội ngũ Khoa học dữ liệu xây dựng hệ gợi ý (*Personalization*) và đội ngũ tăng trưởng người dùng (*User Growth*).
- **Quyết định thực tế được hỗ trợ:** Chuẩn hóa điểm người dùng (*user-mean / leniency normalization*) trong thuật toán gợi ý và thiết kế luồng giới thiệu nội dung riêng cho nhóm *Casual* so với nhóm *Avid/Power*.

## **5. Câu hỏi 5 (Machine Learning — Classification) — Phụ trách: Trần Trọng Tú**

### **5.1. Câu hỏi nghiên cứu và kịch bản dự báo**

#### **5.1.1. Nội dung câu hỏi**
***"Khi một người dùng bắt đầu xem một bộ anime truyền hình (`TV`) đã phát sóng xong, liệu mô hình học máy có thể dự đoán chính xác nguy cơ người dùng đó sẽ **bỏ dở (`Dropped = 1`)** thay vì **hoàn thành (`Completed = 0`)** chỉ dựa trên hồ sơ lịch sử trước đó của người dùng và siêu dữ liệu của bộ anime tại thời điểm bắt đầu hay không, và những đặc trưng nào đóng vai trò quyết định nhất?"***

#### **5.1.2. Đặc tả kỹ thuật tuân thủ 6 yêu cầu ML của đề bài**
1. **Người dùng mô hình & Đặc trưng hợp lệ tại thời điểm dự đoán (*Prediction-time features*):**
   - *Người sử dụng:* Hệ thống gợi ý của nền tảng streaming ngay khoảnh khắc người dùng mở trang phim hoặc bấm tập 1.
   - *Đặc trưng đầu vào:* Thông tin bộ anime (`episodes_clean`, `duration_minutes_clean`, `aired_start_year_clean`, `score_clean`, `log_members`, `source_clean`, `rating_clean`, `franchise_role`, các cờ thể loại) kết hợp với hồ sơ người dùng (`gender_clean`, `user_age_2018`, `account_days_at_crawl`) và lịch sử bỏ dở cá nhân được tính theo cơ chế **Leave-One-Out** (`user_prior_finished_or_dropped`, `user_prior_drop_rate` — đã trừ bỏ chính dòng đang dự đoán!).
   - *Các cột bị loại bỏ hoàn toàn để chống Data Leakage:* `my_watched_episodes_clean` (tiết lộ trực tiếp việc dừng xem trước tập cuối), `my_score_clean` (chấm sau khi xem/bỏ), `my_finish_date_clean`, `my_rewatching_state`, `my_tags_clean`, và các cột tổng `user_dropped` / `user_completed` thô chưa khử dòng mục tiêu.
2. **Chiến lược chia tách dữ liệu (`Group-based Split`):** Nhiều dòng thuộc cùng một `username`. Sử dụng `GroupShuffleSplit` (80% người dùng cho Train, 20% người dùng hoàn toàn mới cho Test) và `GroupKFold(n_splits=4)` trong quá trình huấn luyện để không rò rỉ hành vi cá nhân qua tập kiểm tra.
3. **Mô hình so sánh & `scikit-learn Pipeline`:** Đóng gói toàn bộ xử lý thiếu, chuẩn hóa và mã hóa `OneHotEncoder` trong `ColumnTransformer` + `Pipeline`. So sánh **Baseline tầm thường (`DummyClassifier`)** với **3 mô hình ML**: `LogisticRegression`, `DecisionTreeClassifier` và `HistGradientBoostingClassifier`.
4. **Tối ưu siêu tham số & Thước đo đánh giá:** Tối ưu bằng `GridSearchCV` với `GroupKFold` chỉ trên tập Train; đánh giá trên tập Test đúng 1 lần. Chọn **PR-AUC (`average_precision`)** làm thước đo chính (kèm `ROC-AUC` và `F1-score`) vì lớp `Dropped` chiếm thiểu số (~6–7%), khiến `Accuracy` (~93% khi đoán toàn bộ là `Completed`) gây ngộ nhận nghiêm trọng.
5. **Phân tích lỗi (*Error Analysis*):** Mổ xẻ tỷ lệ dự đoán sai theo nhóm số tập, mức điểm cộng đồng, lịch sử người dùng và kiểm tra các bản ghi lỗi điển hình.
6. **Diễn giải mô hình (*Interpretability*):** Tính `permutation_importance` trên tập Test để lượng hóa vai trò của từng nhóm đặc trưng.

### **5.2. Động lực, đối tượng quan tâm và quyết định thực tế**

- **Đối tượng quan tâm:** Đội ngũ Kỹ sư Hệ thống Gợi ý (*Recommender Systems*) và Quản lý Sản phẩm (*Product Manager*) của dịch vụ streaming.
- **Quyết định thực tế được hỗ trợ:** Điều chỉnh hàm xếp hạng gợi ý theo công thức *Điểm kỳ vọng × $(1 - P(\text{Drop}))$*, tự động ưu tiên các series ngắn gọn dễ hoàn thành cho nhóm người dùng có lịch sử hay bỏ cuộc giữa chừng.

## **6. Câu hỏi 6 (Machine Learning — Regression) — Phụ trách: Phạm Phương**

### **6.1. Câu hỏi nghiên cứu và kịch bản dự báo**

#### **6.1.1. Nội dung câu hỏi**
***"Tại thời điểm tiền sản xuất (trước khi anime phát sóng và chưa có bất kỳ lượt thêm danh sách, lượt yêu thích hay điểm đánh giá nào từ người dùng), chúng ta có thể dự báo mức điểm đánh giá cộng đồng (`score_clean`) của một bộ anime chính xác đến mức nào chỉ từ các đặc trưng tiền sản xuất (`type_clean`, `source_clean`, `episodes_clean`, `duration_minutes_clean`, `rating_clean`, mùa chiếu, `primary_studio`, vị thế `has_prequel`, và tổ hợp `genre`), và đâu là giới hạn dự báo tự nhiên của siêu dữ liệu tĩnh?"***

#### **6.1.2. Đặc tả kỹ thuật tuân thủ 6 yêu cầu ML của đề bài**
1. **Người dùng mô hình & Chống rò rỉ dữ liệu hậu nghiệm (*Post-release Leakage Control*):**
   - *Người sử dụng:* Hội đồng thẩm định mua bản quyền phát sóng quốc tế (*Licensing & Acquisition Team*) trước mùa phim mới.
   - *Các cột bị loại bỏ nghiêm ngặt:* `rank_clean` (hàm trực tiếp của `score`), `members`, `popularity_clean`, `scored_by`, `favorites` (tất cả đều phát sinh sau khi công bố/phát sóng), `status_clean`, `aired_end_year_clean`, và toàn bộ dữ liệu tương tác trong `UserAnimeList_clean`.
2. **Chiến lược chia tách theo thời gian (`Time-based Split`):** Sắp xếp dữ liệu theo năm phát sóng; huấn luyện trên các bộ anime ra mắt **từ năm 2015 trở về trước (`aired_start_year_clean <= 2015`)** và đánh giá khả năng dự báo tương lai trên các bộ anime ra mắt giai đoạn **`2016–2018`**. Trong tập Train, sử dụng **`TimeSeriesSplit(n_splits=5)`** để tune siêu tham số đúng chiều thời gian.
3. **Mô hình so sánh & `scikit-learn Pipeline`:** Toàn bộ xử lý thiếu, chuẩn hóa và `OneHotEncoder(handle_unknown='ignore', min_frequency=15)` đặt trong `Pipeline`. So sánh **Baseline tầm thường (`DummyRegressor` dự đoán trung bình tập Train)** với **3 mô hình hồi quy**: `Ridge`, `RandomForestRegressor` và `HistGradientBoostingRegressor`.
4. **Tối ưu siêu tham số & Thước đo đánh giá:** Chọn **RMSE** làm thước đo tối ưu chính (phạt nặng các sai số thẩm định lớn gây thua lỗ bản quyền), báo cáo song song **MAE** (sai số tuyệt đối trên thang 1–10) và **$R^2$** trên tập Test `2016–2018` đúng 1 lần.
5. **Phân tích lỗi (*Error Analysis*):** Đánh giá phân phối phần dư (`residual`) theo nguyên tác (`Original` so với `Manga` / `Light novel`), theo dải điểm thực tế, và nhận diện các *sleeper hits* (điểm thực vượt xa dự báo) cùng các *bom xịt* giai đoạn 2016–2018.
6. **Diễn giải mô hình (*Interpretability*):** Sử dụng `permutation_importance` trên tập Test `2016–2018` để xác định các tín hiệu tiền sản xuất đáng tin cậy nhất.

### **6.2. Động lực, đối tượng quan tâm và quyết định thực tế**

- **Đối tượng quan tâm:** Bộ phận Mua bản quyền (*Content Acquisition*), nhà phát hành quốc tế và quỹ đầu tư sản xuất anime.
- **Quyết định thực tế được hỗ trợ:** Sàng lọc danh mục dự án tiềm năng trước mùa chiếu và nhận thức rõ biên độ sai số tự nhiên của các dự án kịch bản gốc (`Original`) để thiết lập điều khoản chia sẻ doanh thu linh hoạt thay vì trả phí bảo lãnh cố định quá cao.